# Notebook 02 — Hurricane GIS Tracking
Parses NHC GIS shapefiles and visualises the forecast cone, track, and watch/warning zones.

**Docs:**
- NHC GIS: https://www.nhc.noaa.gov/gis/
- geopandas: https://geopandas.org/en/stable/docs.html

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import geopandas as gpd

from data_fetcher import get_active_storms, parse_hurricane_gis

print('Imports OK')

## 1. Get Active Storms

In [ ]:
storms = get_active_storms()
print(f'{len(storms)} active storm(s) found')
for s in storms:
    print(f"  {s['storm_type']} {s['name']} ({s['storm_id']}) — {s['basin']}")

## 2. Parse GIS Layers for First Active Storm

In [ ]:
if not storms:
    print('No active storms. Cannot demonstrate GIS parsing.')
    gis = {}
    storm = None
else:
    storm = storms[0]
    print(f"Fetching GIS for {storm['storm_type']} {storm['name']}...")
    gis = parse_hurricane_gis(storm)

    for layer_name, gdf in gis.items():
        if gdf is not None:
            print(f'  [{layer_name}] — {len(gdf)} features, CRS: {gdf.crs}')
        else:
            print(f'  [{layer_name}] — not available')

## 3. Visualise Forecast Track & Cone
*(Requires an active storm with GIS data)*

In [ ]:
if not storm or all(v is None for v in gis.values()):
    print('No GIS data available to plot. Skipping visualisation.')
else:
    fig, ax = plt.subplots(figsize=(12, 8))

    # Natural Earth basemap (included with geopandas)
    world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
    world.plot(ax=ax, color='#f0f0f0', edgecolor='#aaaaaa', linewidth=0.5)

    legend_handles = []

    if gis.get('cone_polygon') is not None:
        gis['cone_polygon'].plot(ax=ax, alpha=0.25, color='steelblue', label='5-day Cone')
        legend_handles.append(mpatches.Patch(color='steelblue', alpha=0.5, label='5-day Cone'))

    if gis.get('track_line') is not None:
        gis['track_line'].plot(ax=ax, color='darkred', linewidth=2, label='Track')
        legend_handles.append(mpatches.Patch(color='darkred', label='Forecast Track'))

    if gis.get('track_points') is not None:
        gis['track_points'].plot(ax=ax, color='red', markersize=60, zorder=5, marker='o')

    if gis.get('watches_warnings') is not None:
        gis['watches_warnings'].plot(ax=ax, color='orange', alpha=0.5, label='Watch/Warning')
        legend_handles.append(mpatches.Patch(color='orange', alpha=0.7, label='Watch/Warning Zone'))

    # Set map extent to storm area (Gulf / Atlantic)
    ax.set_xlim(-105, -60)
    ax.set_ylim(10, 45)
    ax.set_title(
        f"{storm['storm_type']} {storm['name']} — 5-day Forecast Track & Cone",
        fontsize=14, fontweight='bold'
    )
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.legend(handles=legend_handles, loc='lower left')
    plt.tight_layout()
    plt.savefig(f"{storm['storm_id']}_track.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Map saved as {storm['storm_id']}_track.png")

## 4. Inspect GIS Attribute Table

In [ ]:
for layer_name, gdf in gis.items():
    if gdf is not None and not gdf.empty:
        print(f'\n=== {layer_name} ===')
        print(gdf.drop(columns='geometry').head(5).to_string())